# 16.13 分塊處理，是否代表看不到其他塊？


整理六張卡片時，你可以每次拿三張出來處理，最後仍讀完六張。也可以規定每張卡片只能和同組的卡片互相參照。兩種做法都叫「分塊」，卻改變不同的東西：前者安排工作的先後，後者限制資料之間的連結。

注意力也有這個差別。[逐塊累積加權結果](https://birdhackor.github.io/tiny-perceptron-vlm/16.9.html)時，我們把候選拆成兩塊，最後仍讓同一個 Query 用到所有允許的 Key、Value。分塊並沒有把後一塊的內容排除。現在把這件事和上一節的滑動視窗放在同一張配對圖中看。

六個位置從 0 編到 5。圖的一列代表「正在查資料的位置」，一欄代表「可以讀的位置」；藍格表示允許讀取，灰格表示不能讀，藍格和 16.5、16.8 圖中的綠格意思相同。三種規則都禁止偷看未來，因此對角線右上方都是灰色。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-16-13-visible-pairs.svg")))

第一張表採**完整因果注意力**。位置 3 能讀 0、1、2、3。即使把計算安排成先處理 0–2、再處理 3–5，所有藍格仍要算到；完成每一列所需的統計後，才得到這一列的結果。FlashAttention 的分塊運算可以在相同的因果規則下保留這些連結，減少大中間表的保存和記憶體搬移。

第二張表採**滑動視窗**（圖上簡寫為「滑窗」），寬度為 3，包含目前位置。位置 3 只讀 1、2、3；位置 4 只讀 2、3、4。視窗隨著位置移動，因此能跨越我們為計算方便劃出的 0–2 和 3–5 邊界。

第三張表採本節定義的**區塊局部規則**：0–2 是一組，3–5 是另一組，每個位置只讀同組中自己和更早的位置。位置 3 是第二組的開頭，只能讀自己；位置 4 能讀 3、4。這個例子沒有另外加入跨組通路，不能讀取前一組的內容。其他區塊架構可以增加全域位置或交錯分組，必須依其實際規則重新畫圖。

三張表都可以用 [16.5](https://birdhackor.github.io/tiny-perceptron-vlm/16.5.html) 的 `attention_mask` 組出來：完整因果只傳位置；滑動視窗再加上「往回的距離小於 3」；區塊局部則把每三個位置編成同一組，交給 `segments`。


In [ ]:
import torch
from tiny_perceptron.attention import attention_mask

position = torch.arange(6)
query = 3
causal = attention_mask(position, position)[0, 0]
sliding = causal & (position[:, None] - position[None, :] < 3)
group = position // 3
block_local = attention_mask(position, position, segments=group[None])[0, 0]
for name, rule in (("完整因果", causal), ("滑動視窗", sliding), ("區塊局部", block_local)):
    print(name, "允許格數", rule.sum().item(), "位置", query, "可讀", position[rule[query]].tolist())

`attention_mask(position, position)` 只用因果條件，`[0, 0]` 取出唯一一段、唯一一頭的六乘六表。`position[:, None] - position[None, :]` 用廣播算出「Query 位置減 Key 位置」，也就是往回的距離；`< 3` 只留距離 0、1、2，對上寬度 3、包含目前位置的約定，`&` 再和因果條件取交集。`position // 3` 是 [W.2](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2) 的整數除法，把位置 0、1、2 編成第 0 組，3、4、5 編成第 1 組；`[None]` 補上筆數軸後交給 `segments`。迴圈對三張表各印兩件事：`rule.sum().item()` 數出 True 的格數（True 算 1），`position[rule[query]]` 用 `query` 那一列（這裡是位置 3）的真假值挑出可讀的位置。輸出依序是 21 格與 `[0, 1, 2, 3]`、15 格與 `[1, 2, 3]`、12 格與 `[3]`，和圖一致。

所以區塊局部和 16.5 的文件隔離是同一種遮罩：因果，而且同組。差別在用途：16.5 的兩組是互不相干的文件；這裡是同一段文字被切開，前一組的內容本來可能有用，只是這個規則不讓直接讀。

同樣看位置 3，三種規則便很容易分辨：

| 規則 | 位置 3 直接讀哪裡？ | 改變了什麼？ |
| --- | --- | --- |
| 完整因果，可逐塊計算 | 0、1、2、3 | 計算安排可以改，這些連結仍保留 |
| 滑動視窗寬度 3 | 1、2、3 | 限制直接查看的距離 |
| 每三個位置一組的區塊局部 | 3 | 限制直接查看的分組 |

第一張表的藍格仍有 21 格要算，滑動視窗只有 15 格，這個區塊局部例子是 12 格。逐塊計算完整因果注意力可以避免保存完整分數表，但並沒有因此把要算的配對數降成後兩者。完整注意力的平方級工作量，和所需中間儲存的大小，也要分開談。

實務上，kernel（計算程式）和可見規則可以各自選擇。支援某種遮罩的 FlashAttention kernel 也可能用來執行滑動視窗；看到「Flash」這個名稱，不能直接判斷模型是不是能讀全部前文。反過來，寫了局部遮罩也不表示程式已用省成本的局部 kernel：上面的 `attention_mask` 仍建立完整的六乘六表。判讀一個設計時，先問「哪些格允許相連」，再問「程式如何計算那些格」，就能把能力範圍和效率分清楚。

練習改這段程式，每步改完都從頭執行。第一步把 `query = 3` 改成 `query = 4`，先預測三種規則下位置 4 各能讀哪些位置；答案分別是 0–4、2–4、3–4。第二步改回 `query = 3`，再把 `group = position // 3` 改成 `group = position // 2`，讓區塊局部每兩個位置一組：先預測只有區塊局部那一行改變，允許格數由 12 變成 9，位置 3 改讀 2、3，完整因果與滑動視窗不變。第三步不必執行：如果只是把計算切成更小的塊，例如 [16.9](https://birdhackor.github.io/tiny-perceptron-vlm/16.9.html) 練習把每塊改成 1 個候選，三張表會不會變？都不會。計算的分塊只改變工作的先後；可讀範圍只由規則決定，第二步改的正是區塊局部規則本身。

<details>
<summary>來源與延伸</summary>

[FlashAttention 原論文第 3.1–3.2 節](https://arxiv.org/pdf/2205.14135v2)說明完整注意力的分塊、統計合併和記憶體搬移；其區塊稀疏延伸是另一種可見規則。此處的「完整」仍遵守本課的因果遮罩，沒有允許讀未來。不同運算順序可能產生浮點尾數差異，exact 不表示逐 bit 相同。

[Transformers v4.57.1 的遮罩實作](https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/masking_utils.py)分別定義完整、滑動視窗與分組等規則。本節第三張表是每三個位置一組的簡化設計，不代表所有名稱含 chunk 的實作。

</details>
